[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camiloandcu/Retail-Demand-Forecasting/blob/main/01_data_audit_eda.ipynb)

# 01 - Data acquisition, validation, and exploration

**Purpose.** Acquire the Corporación Favorita competition files, verify their integrity and schema, and identify data properties that affect a 16-day forecast. No model is trained here.

**Inputs:** `configs/full.yaml`, seven Kaggle CSV files, and the reusable `retail_forecast` package. Set the environment variable `RETAIL_FORECAST_MODE=smoke` to use deterministic schema-only fixtures.

**Outputs:** `artifacts/metrics/eda_summary.json`, `artifacts/metrics/dataset_manifest.json`, and up to nine PNG files under `artifacts/figures/eda/`. Raw files are never modified.

**Expected resources:** CPU runtime; at least 2 GB RAM. Full mode usually takes 3–8 minutes after download; smoke mode takes less than one minute. Times depend on the Colab runtime and network.

## Reading guide

The notebook labels statements as **OBSERVATION** (directly measured), **INFERENCE** (a design consequence), or **HYPOTHESIS** (a claim that still requires temporal validation). All sales analysis uses `train.csv`; test targets do not exist and no random split is used.

In [ ]:
# Clone only when the notebook is opened in a fresh Colab runtime.
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    REPO_ROOT = Path("/content/Retail-Demand-Forecasting")
    if not (REPO_ROOT / "pyproject.toml").is_file():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        subprocess.run(
            ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
else:
    REPO_ROOT = Path.cwd().resolve()
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise RuntimeError("Run the notebook from the repository root.")

os.chdir(REPO_ROOT)
if importlib.util.find_spec("pip") is not None:
    install_command = [sys.executable, "-m", "pip", "install", "-e", ".[notebook]"]
elif shutil.which("uv"):
    install_command = ["uv", "pip", "install", "--python", sys.executable, "-e", ".[notebook]"]
else:
    raise RuntimeError("This environment provides neither pip nor uv.")
subprocess.run(install_command, check=True)
source_path = str(REPO_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
if importlib.util.find_spec("retail_forecast") is None:
    raise RuntimeError("retail_forecast was installed but is not importable.")
print(f"Repository ready at {REPO_ROOT} with Python {sys.version.split()[0]}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from retail_forecast import __version__
from retail_forecast.acquisition import download_competition_data
from retail_forecast.config import load_config
from retail_forecast.data import required_files_exist, validate_dataset, write_synthetic_dataset
from retail_forecast.eda import (
    build_eda_summary,
    build_eda_tables,
    build_manifest,
    export_eda_artifacts,
    load_eda_frames,
    save_figure,
    validate_kaggle_hashes,
)
from retail_forecast.logging_utils import configure_logging
from retail_forecast.reproducibility import set_global_seed
from retail_forecast.versions import capture_versions

MODE = os.getenv("RETAIL_FORECAST_MODE", "full").strip().lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("RETAIL_FORECAST_MODE must be 'full' or 'smoke'.")
config = load_config(f"configs/{MODE}.yaml")
configure_logging(config.runtime.log_level)
set_global_seed(config.runtime.seed)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.titleweight": "bold"})
print(f"mode={MODE} | package={__version__} | horizon={config.forecast.horizon}")
display(capture_versions())

## Secure Kaggle acquisition

In Colab, open the key icon (**Secrets**), create a secret named `KAGGLE_API_TOKEN`, paste the token generated at Kaggle Settings → API, and grant notebook access. The next cell reads it only when a download is necessary, never prints it, never creates `kaggle.json`, and removes the environment variable afterward. Locally, provide the same environment variable before starting Jupyter.

You must accept the competition rules on Kaggle before the first download. Existing complete files are reused.

In [ ]:
if config.data.source == "synthetic":
    if not required_files_exist(config.paths.raw_data_dir):
        write_synthetic_dataset(config)
else:
    if not required_files_exist(config.paths.raw_data_dir):
        token_loaded = False
        if IN_COLAB:
            from google.colab import userdata

            kaggle_token = userdata.get("KAGGLE_API_TOKEN")
            if not kaggle_token:
                raise RuntimeError("Add KAGGLE_API_TOKEN to Colab Secrets and grant access.")
            os.environ["KAGGLE_API_TOKEN"] = kaggle_token
            token_loaded = True
        try:
            download_competition_data(config.paths.raw_data_dir)
        finally:
            if token_loaded:
                os.environ.pop("KAGGLE_API_TOKEN", None)
                del kaggle_token

hashes = validate_kaggle_hashes(config.paths.raw_data_dir) if MODE == "full" else {}
dataset_contract = validate_dataset(config)
print("Acquisition and integrity checks: PASS")
display(dataset_contract.to_dict())

In [ ]:
frames = load_eda_frames(config.paths.raw_data_dir)
manifest = build_manifest(config.paths.raw_data_dir, frames)
tables = build_eda_tables(frames)
eda_summary = build_eda_summary(frames, tables, config.data.source)

METRICS_DIR = REPO_ROOT / "artifacts/metrics"
FIGURES_DIR = REPO_ROOT / "artifacts/figures/eda"
summary_path, manifest_path = export_eda_artifacts(eda_summary, manifest, METRICS_DIR)
manifest_view = pd.DataFrame(
    [
        {
            "file": filename,
            "bytes": values["bytes"],
            "rows": values["rows"],
            "columns": len(values["columns"]),
            "missing": sum(values["missing"].values()),
            "date_min": values.get("date_min"),
            "date_max": values.get("date_max"),
        }
        for filename, values in manifest.items()
    ]
)
display(manifest_view)
print(f"Detailed dtypes and missingness exported to {manifest_path}")

## Question 1 - How does total demand evolve over time, and is there evidence of level changes?

In [ ]:
daily = tables["daily_sales"]
fig, ax = plt.subplots()
ax.plot(daily["date"], daily["sales"], alpha=0.35, linewidth=0.8, label="Daily total")
ax.plot(daily["date"], daily["rolling_28d"], linewidth=2, label="28-day median")
ax.set(title="Daily sales and local level", xlabel="Date", ylabel="Units")
ax.legend()
save_figure(fig, FIGURES_DIR, "01_daily_sales.png")
plt.show()

In [ ]:
ratio = eda_summary["observations"]["last_to_first_90d_ratio"]
display(
    Markdown(
        f"**Conclusion — OBSERVATION:** the last/first 90-day mean ratio is "
        f"**{ratio:.3f}**. **INFERENCE:** rolling-origin folds and recent-window "
        "features are safer than assuming a fixed level."
    )
)

## Question 2 - How many zeros and extreme values does the target contain?

In [ ]:
sales = frames["train.csv"]["sales"]
fig, ax = plt.subplots()
ax.hist(np.log1p(sales), bins=60, color="#4472C4", alpha=0.85)
ax.set(title="Distribution of log1p(sales)", xlabel="log1p(sales)", ylabel="Rows")
save_figure(fig, FIGURES_DIR, "02_sales_distribution.png")
plt.show()

In [ ]:
obs = eda_summary["observations"]
display(
    Markdown(
        f"**Conclusion — OBSERVATION:** zeros represent "
        f"**{obs['zero_sales_rate']:.2%}** of rows and "
        f"**{obs['iqr_outlier_candidates']:,}** rows exceed the IQR rule. "
        "**INFERENCE:** keep zeros as valid targets and use RMSLE/log-scale "
        "diagnostics; extreme values are candidates for investigation, not deletion."
    )
)

## Question 3 - How uneven is demand across stores?

In [ ]:
store_sales = tables["store_sales"].sort_values("sales")
fig, ax = plt.subplots()
ax.barh(store_sales["store_nbr"].astype(str), store_sales["sales"], color="#70AD47")
ax.set(title="Total sales by store", xlabel="Units", ylabel="Store")
save_figure(fig, FIGURES_DIR, "03_store_sales.png")
plt.show()

In [ ]:
store_ratio = store_sales["sales"].max() / max(store_sales["sales"].min(), 1e-12)
display(
    Markdown(
        "**Conclusion — OBSERVATION:** the largest-to-smallest store total ratio is "
        f"**{store_ratio:.2f}**. **HYPOTHESIS:** store identity or metadata may "
        "explain persistent scale differences; test this inside temporal folds."
    )
)

## Question 4 - Which families dominate sales, and where are zeros concentrated?

In [ ]:
family_sales = tables["family_sales"].head(15).sort_values("sales")
fig, ax = plt.subplots()
ax.barh(family_sales["family"], family_sales["sales"], color="#ED7D31")
ax.set(title="Top families by total sales", xlabel="Units", ylabel="Family")
save_figure(fig, FIGURES_DIR, "04_family_sales.png")
plt.show()

In [ ]:
all_family = tables["family_sales"]
highest_zero = all_family.sort_values("zero_rate", ascending=False).iloc[0]
display(
    Markdown(
        f"**Conclusion — OBSERVATION:** `{highest_zero['family']}` has the highest "
        f"measured zero rate (**{highest_zero['zero_rate']:.2%}**). **INFERENCE:** "
        "report family-level errors so high-volume families do not hide intermittent series."
    )
)

## Question 5 - Are promoted rows associated with different sales and zero rates?

In [ ]:
promotion = tables["promotion"].copy()
labels = promotion["promoted"].map({False: "No promotion", True: "Promoted"})
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(labels, promotion["mean_sales"], color=["#A5A5A5", "#FFC000"])
axes[0].set(title="Mean sales", ylabel="Units")
axes[1].bar(labels, promotion["zero_rate"], color=["#A5A5A5", "#FFC000"])
axes[1].set(title="Zero-sales rate", ylabel="Share")
fig.suptitle("Promotion association")
save_figure(fig, FIGURES_DIR, "05_promotions.png")
plt.show()

In [ ]:
promo_obs = promotion.set_index("promoted")
lift = promo_obs.loc[True, "mean_sales"] / max(promo_obs.loc[False, "mean_sales"], 1e-12)
display(
    Markdown(
        "**Conclusion — OBSERVATION:** promoted rows have a mean-sales ratio of "
        f"**{lift:.2f}×** versus non-promoted rows. **HYPOTHESIS:** promotion is "
        "predictive, but this is not a causal estimate. Future `onpromotion` is "
        "allowed because Kaggle supplies it."
    )
)

## Question 6 - Do daily transaction counts move with aggregate sales?

In [ ]:
sales_transactions = tables["sales_transactions"]
fig, ax = plt.subplots()
ax.scatter(sales_transactions["transactions"], sales_transactions["sales"], s=12, alpha=0.35)
ax.set(title="Daily transactions and sales", xlabel="Transactions", ylabel="Units sold")
save_figure(fig, FIGURES_DIR, "06_transactions.png")
plt.show()

In [ ]:
transaction_corr = sales_transactions[["transactions", "sales"]].corr().iloc[0, 1]
coverage = eda_summary["observations"]["transaction_store_date_coverage"]
display(
    Markdown(
        "**Conclusion — OBSERVATION:** aggregate Pearson correlation is "
        f"**{transaction_corr:.3f}** and store-date coverage is **{coverage:.2%}**. "
        "**INFERENCE:** only lagged transactions are eligible because test has no "
        "future transactions; missing rows are not zeros."
    )
)

## Question 7 - Is oil price associated with aggregate sales over the training period?

In [ ]:
oil_sales = tables["oil_sales"].dropna(subset=["dcoilwtico"])
fig, ax = plt.subplots()
ax.scatter(oil_sales["dcoilwtico"], oil_sales["sales"], s=12, alpha=0.35, color="#5B9BD5")
ax.set(title="Oil price and daily sales", xlabel="WTI price", ylabel="Units sold")
save_figure(fig, FIGURES_DIR, "07_oil.png")
plt.show()

In [ ]:
oil_corr = oil_sales[["dcoilwtico", "sales"]].corr().iloc[0, 1]
oil_missing = eda_summary["observations"]["oil_missing_values"]
display(
    Markdown(
        "**Conclusion — OBSERVATION:** contemporaneous aggregate correlation is "
        f"**{oil_corr:.3f}** with **{oil_missing}** missing oil values. **HYPOTHESIS:** "
        "predictive value may be indirect; use only Kaggle-provided horizon values "
        "and preserve a missingness indicator."
    )
)

## Question 8 - What holiday/event types exist, and can a date-only join be trusted?

In [ ]:
holiday_types = tables["holiday_types"].sort_values("size")
fig, ax = plt.subplots()
ax.barh(holiday_types["type"], holiday_types["size"], color="#8064A2")
ax.set(title="Holiday and event records by type", xlabel="Rows", ylabel="Type")
save_figure(fig, FIGURES_DIR, "08_holidays.png")
plt.show()

In [ ]:
duplicates = eda_summary["observations"]["holiday_duplicate_date_rows"]
display(
    Markdown(
        f"**Conclusion — OBSERVATION:** **{duplicates}** holiday rows share their date "
        "with another record. **INFERENCE:** resolve national/regional/local scope and "
        "transfer semantics before joining; a raw date join can multiply sales rows."
    )
)

## Question 9 - What weekly, monthly, and annual seasonal structure is visible using train only?

In [ ]:
weekday = tables["weekday"]
month = tables["month"]
annual_cycle = tables["annual_cycle"]
weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar([weekday_names[int(value)] for value in weekday["weekday"]], weekday["sales"])
axes[0].set(title="Weekly", ylabel="Mean daily units")
axes[1].bar(month["month"].astype(str), month["sales"], color="#70AD47")
axes[1].set(title="Monthly", xlabel="Month")
axes[2].plot(annual_cycle["dayofyear"], annual_cycle["sales"], color="#C00000")
axes[2].set(title="Annual cycle", xlabel="Day of year")
fig.suptitle("Seasonal patterns measured from training data only")
save_figure(fig, FIGURES_DIR, "09_seasonality.png")
plt.show()

In [ ]:
best_weekday = weekday.loc[weekday["sales"].idxmax(), "weekday"]
best_month = month.loc[month["sales"].idxmax(), "month"]
best_annual_day = annual_cycle.loc[annual_cycle["sales"].idxmax(), "dayofyear"]
display(
    Markdown(
        "**Conclusion — OBSERVATION:** the largest mean occurs on "
        f"**{weekday_names[int(best_weekday)]}**, month **{int(best_month)}**, and "
        f"day-of-year **{int(best_annual_day)}**. **INFERENCE:** retain calendar "
        "covariates, but estimate learned transformations inside each temporal fold."
    )
)

## Final checks and model-design consequences

In [ ]:
final_manifest = build_manifest(config.paths.raw_data_dir, frames)
raw_unchanged = all(manifest[name]["sha256"] == final_manifest[name]["sha256"] for name in manifest)
figure_files = sorted(FIGURES_DIR.glob("*.png"))
checks = {
    "16-day horizon": dataset_contract.horizon == 16,
    "seven schemas loaded": len(frames) == 7,
    "Kaggle hashes verified or synthetic mode": MODE == "smoke" or len(hashes) == 7,
    "raw files unchanged during EDA": raw_unchanged,
    "EDA summary exported": summary_path.is_file(),
    "dataset manifest exported": manifest_path.is_file(),
    "figure limit respected": 0 < len(figure_files) <= 10,
    "future transactions absent": frames["transactions.csv"]["date"].max()
    <= frames["train.csv"]["date"].max(),
}
check_table = pd.DataFrame(
    [{"check": name, "status": "PASS" if passed else "FAIL"} for name, passed in checks.items()]
)
display(check_table)
if not all(checks.values()):
    raise AssertionError(
        f"EDA checks failed: {[name for name, passed in checks.items() if not passed]}"
    )
print(f"PASS: {len(checks)} checks; {len(figure_files)} figures; raw inputs unchanged.")

In [ ]:
display(Markdown("### Observations"))
display(eda_summary["observations"])
display(Markdown("### Inferences that affect model design"))
for item in eda_summary["inferences"] + eda_summary["model_design_effects"]:
    display(Markdown(f"- {item}"))
display(Markdown("### Hypotheses to test with rolling-origin validation"))
for item in eda_summary["hypotheses"]:
    display(Markdown(f"- {item}"))
print(f"Artifacts ready: {summary_path} and {FIGURES_DIR}")